In [13]:
import mlflow

from sarah_hotel_reservation_data.data_processing import HotelDataset

mlflow.set_tracking_uri("databricks")
mlflow.set_experiment(experiment_name="/Shared/hotel-data-sarah")
mlflow.set_experiment_tags({"repository_name": "hotel-data-sarah"})

hotel_data = HotelDataset(
    data_filepath="../data/Hotel Reservations.csv",
    yaml_file_path="../hotel_project_config.yaml",
)
hotel_data.run_data_preparation()
X_train, y_train = hotel_data.get_train_data()
X_val, y_val = hotel_data.get_val_data()

X_train.head()

## Model Experiments / offline

In [15]:
import pandas as pd
import yaml
from scipy.stats import randint
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [22]:
type(X_val)

pandas.core.frame.DataFrame

In [25]:
with open("../hotel_project_config.yaml", "r") as yaml_file:
    config = yaml.safe_load(yaml_file)


numeric_features = config["numerical_columns"]
categorical_features = config["categorical_columns"]

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(drop="first")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("randomforestclassifier", RandomForestClassifier(random_state=42)),
    ]
)

param_space = {
    "randomforestclassifier__n_estimators": randint(200, 600),
    "randomforestclassifier__max_depth": randint(30, 60),
    "randomforestclassifier__min_samples_split": randint(2, 10),
    "randomforestclassifier__min_samples_leaf": randint(1, 4),
}

random_search = RandomizedSearchCV(
    pipeline,
    param_space,
    n_iter=50,
    cv=5,
    scoring="f1",
    random_state=42,
    error_score="raise",
    n_jobs=-1,
)

random_search.fit(X_train, y_train.values.ravel())

# Get best model & predictions
best_model = random_search.best_estimator_
y_pred = best_model.predict(X_val)

# Calculate metrics
metrics = {
    "accuracy": accuracy_score(y_val, y_pred),
    "precision": precision_score(y_val, y_pred, average="binary"),
    "recall": recall_score(y_val, y_pred, average="binary"),
    "f1": f1_score(y_val, y_pred, average="binary"),
}

# Print results
print("Model Evaluation Metrics on Validation Set:")
for metric_name, value in metrics.items():
    print(f"{metric_name.capitalize()}: {value:.4f}")

# Get the best parameters from the random search
best_params = random_search.best_params_
print("\nBest Model Parameters:")
for param, value in best_params.items():
    # Remove the pipeline prefix for cleaner output
    clean_param = param.replace("randomforestclassifier__", "")
    print(f"{clean_param}: {value}")

/Users/sarahglasmacher/Documents/Coding/marvelous-databricks-course-GalaxyInfernoCodes/.venv/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Model Evaluation Metrics on Validation Set:
Accuracy: 0.8983
Precision: 0.8795
Recall: 0.8036
F1: 0.8398

Best Model Parameters:
max_depth: 56
min_samples_leaf: 1
min_samples_split: 3
n_estimators: 430


In [26]:
with open("../hotel_project_config.yaml", "r") as yaml_file:
    config = yaml.safe_load(yaml_file)


class TotalNightsTransformer(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self  # No fitting necessary

    def transform(self, X):
        # Ensure X is a DataFrame
        if isinstance(X, pd.DataFrame):
            # Create the new feature
            X["no_of_total_nights"] = X["no_of_weekend_nights"] + X["no_of_week_nights"]
        return X


numeric_features = config["numerical_columns"] + ["no_of_total_nights"]
categorical_features = config["categorical_columns"]

numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(drop="first")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

full_preprocessor = Pipeline(
    [
        (
            "total_nights_transformer",
            TotalNightsTransformer(),
        ),  # Add the custom transformer
        ("feature_preprocessing", preprocessor),
    ]
)

pipeline = Pipeline(
    [
        ("preprocessor", full_preprocessor),
        ("randomforestclassifier", RandomForestClassifier(random_state=42)),
    ]
)

param_space = {
    "randomforestclassifier__n_estimators": randint(200, 600),
    "randomforestclassifier__max_depth": randint(30, 60),
    "randomforestclassifier__min_samples_split": randint(2, 10),
    "randomforestclassifier__min_samples_leaf": randint(1, 4),
}

random_search = RandomizedSearchCV(
    pipeline, param_space, n_iter=50, cv=5, scoring="f1", random_state=42, n_jobs=-1
)

random_search.fit(X_train, y_train.values.ravel())

# Get best model & predictions
best_model = random_search.best_estimator_
y_pred = best_model.predict(X_val)

# Calculate metrics
metrics = {
    "accuracy": accuracy_score(y_val, y_pred),
    "precision": precision_score(y_val, y_pred, average="binary"),
    "recall": recall_score(y_val, y_pred, average="binary"),
    "f1": f1_score(y_val, y_pred, average="binary"),
}

# Print results
print("Model Evaluation Metrics on Validation Set:")
for metric_name, value in metrics.items():
    print(f"{metric_name.capitalize()}: {value:.4f}")

# Get the best parameters from the random search
best_params = random_search.best_params_
print("\nBest Model Parameters:")
for param, value in best_params.items():
    # Remove the pipeline prefix for cleaner output
    clean_param = param.replace("randomforestclassifier__", "")
    print(f"{clean_param}: {value}")

/Users/sarahglasmacher/Documents/Coding/marvelous-databricks-course-GalaxyInfernoCodes/.venv/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:752: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Model Evaluation Metrics on Validation Set:
Accuracy: 0.8982
Precision: 0.8785
Recall: 0.8042
F1: 0.8397

Best Model Parameters:
max_depth: 30
min_samples_leaf: 1
min_samples_split: 3
n_estimators: 454


In [7]:
# Define the experiment tracking function
def train_and_log_model(X_train, X_val, y_train, y_val):
    with mlflow.start_run():
        param_space = {
            "n_estimators": randint(100, 500),
            "max_depth": randint(10, 50),
            "min_samples_split": randint(2, 10),
            "min_samples_leaf": randint(1, 4),
        }

        # Initialize model with RandomizedSearchCV
        rf = RandomForestClassifier(random_state=42)
        random_search = RandomizedSearchCV(
            rf, param_space, n_iter=10, cv=5, scoring="f1", random_state=42, verbose=2
        )

        random_search.fit(X_train, y_train.values.ravel())

        # Log best parameters
        mlflow.log_params(random_search.best_params_)

        # Get best model
        best_model = random_search.best_estimator_

        # Generate predictions
        y_pred = best_model.predict(X_val)

        # Calculate metrics
        metrics = {
            "accuracy": accuracy_score(y_val, y_pred),
            "precision": precision_score(y_val, y_pred, average="binary"),
            "recall": recall_score(y_val, y_pred, average="binary"),
            "f1": f1_score(y_val, y_pred, average="binary"),
        }

        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(best_model, "random_forest_model")

        return best_model, metrics


# Run the experiment
best_model, metrics = train_and_log_model(X_train, X_val, y_train, y_val)

# Print results
print("Model Evaluation Metrics on Validation Set:")
for metric_name, value in metrics.items():
    print(f"{metric_name.capitalize()}: {value:.4f}")

Fitting 5 folds for each of 10 candidates, totalling 50 fits
[CV] END max_depth=48, min_samples_leaf=1, min_samples_split=8, n_estimators=206; total time=   1.7s
[CV] END max_depth=48, min_samples_leaf=1, min_samples_split=8, n_estimators=206; total time=   1.7s
[CV] END max_depth=48, min_samples_leaf=1, min_samples_split=8, n_estimators=206; total time=   1.7s
[CV] END max_depth=48, min_samples_leaf=1, min_samples_split=8, n_estimators=206; total time=   1.7s
[CV] END max_depth=48, min_samples_leaf=1, min_samples_split=8, n_estimators=206; total time=   1.7s
[CV] END max_depth=17, min_samples_leaf=1, min_samples_split=6, n_estimators=202; total time=   1.5s
[CV] END max_depth=17, min_samples_leaf=1, min_samples_split=6, n_estimators=202; total time=   1.5s
[CV] END max_depth=17, min_samples_leaf=1, min_samples_split=6, n_estimators=202; total time=   1.5s
[CV] END max_depth=17, min_samples_leaf=1, min_samples_split=6, n_estimators=202; total time=   1.5s
[CV] END max_depth=17, min_sam

2024/11/22 13:54:11 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2024/11/22 13:54:42 INFO mlflow.tracking._tracking_service.client: 🏃 View run monumental-fowl-736 at: https://adb-3462382944381210.10.azuredatabricks.net/ml/experiments/688553492365940/runs/c5a6401371604d1fae60f0df05cd7d5b.
2024/11/22 13:54:42 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: https://adb-3462382944381210.10.azuredatabricks.net/ml/experiments/688553492365940.


Model Evaluation Metrics on Validation Set:
Accuracy: 0.8982
Precision: 0.8807
Recall: 0.8016
F1: 0.8393


In [8]:
clf = RandomForestClassifier()
clf.fit(X_train, y_train)

/Users/sarahglasmacher/Documents/Coding/marvelous-databricks-course-GalaxyInfernoCodes/.venv/lib/python3.11/site-packages/sklearn/base.py:1473: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


RandomForestClassifier()

In [9]:
# 1. Generate predictions on validation set
y_pred = clf.predict(X_val)

# 2. Calculate evaluation metrics
accuracy = accuracy_score(y_val, y_pred)
precision = precision_score(
    y_val, y_pred, average="binary"
)  # 'binary' for binary classification
recall = recall_score(y_val, y_pred, average="binary")
f1 = f1_score(y_val, y_pred, average="binary")

# 3. Print evaluation results
print("Model Evaluation Metrics on Validation Set:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

# 4. Display the confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_val, y_pred))

# 5. Detailed classification report
print("\nClassification Report:")
print(classification_report(y_val, y_pred))

Model Evaluation Metrics on Validation Set:
Accuracy: 0.8983
Precision: 0.8825
Recall: 0.8000
F1 Score: 0.8392

Confusion Matrix:
[[3674  205]
 [ 385 1540]]

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.95      0.93      3879
           1       0.88      0.80      0.84      1925

    accuracy                           0.90      5804
   macro avg       0.89      0.87      0.88      5804
weighted avg       0.90      0.90      0.90      5804

